<img src="https://www.hazfundacion.org/wp-content/uploads/2022/11/U-Carlos-III.png" width="300" alt="Universidad Carlos III de Madrid">

**Máster Universitario en Robótica y Automatización**

Asignatura: Procesamiento de Imágenes por Computador

Curso: 2026-27

Autores:
- Anthony Manzueta Difo
- Stefan Oshchypok
- Marina Ruiz Izu

Proyecto: **Reconocimiento de ocupación de plazas de aparcamiento mediante fine-tuning de redes preentrenadas**

# Problema a tratar

## Motivación

Conocer la disponibilidad de plazas libres es útil para la gestión de los aparcamientos y para reducir el tiempo y el tráfico que genera la búsqueda de plaza. Con la ocupación en tiempo real, los conductores ahorran tiempo de búsqueda.

Desde el punto de vista empresarial, la visión por computador aprovecha las cámaras de seguridad ya instaladas. Evita el coste de un sensor por plaza (en torno a 300-600 $ por plaza en los sistemas tradicionales) y la obra asociada. Además de la ocupación en tiempo real, permite obtener datos de uso (ocupación por franja horaria, tiempos de estancia) que ayudan a dimensionar y optimizar el aparcamiento. En empresas con parking en sus oficinas, mejora la experiencia de los empleados y reduce el tiempo y las emisiones de la búsqueda de plaza.

## Objetivo

Detectar plazas de aparcamiento libres y ocupadas mediante una cámara exterior. Dada una imagen de aparcamiento, el sistema localiza cada plaza y la clasifica como libre u ocupada, devolviendo para cada una un valor de confianza.

# 1. Entorno

## 1.1 Imports

In [12]:
# !pip install -q huggingface_hub
#!pip install -q ultralytics

import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from huggingface_hub import snapshot_download

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## 1.2 Configuración

In [3]:
IN_COLAB = 'google.colab' in sys.modules
DATA = '/content/data' if IN_COLAB else '../data'
OUTPUT = '/content/results/pretrained_models' if IN_COLAB else '../results/pretrained_models'


CONFIG = {
    'dataset': {
        'repo': 'StafLoker/Meta-PKLot-mini',
        'path': f'{DATA}/Meta-PKLot-mini'
    },
    'test': {
        'file': 'test.txt', 
        'local_dir': '../data_split',
        'url': 'https://raw.githubusercontent.com/StafLoker/pic-proyecto/main/data_split',
    },
    'models': {
        'yolo11m':   {'weights': 'yolo11m.pt', 'imgsz': 1280},
        'retinanet': {'weights': 'DEFAULT'},
        'maskrcnn':  {'weights': 'DEFAULT'},
    },
    'inference': {
        'score_threshold': 0.25, # Confianza mínima para aceptar detección
        'vehicle_classes': ['car', 'truck', 'bus'], # clases de COCO que cuentan como ocupa plaza
    },
    'occupancy': { # Fracción de área de la plaza que tiene que cubrir el vehiculo para estar ocupada
        'min_overlap': 0.3,
    },
    'seed': 99,
}

os.makedirs(OUTPUT, exist_ok=True)

# 2. Dataset

## 2.1 Descarga

Descarga Meta-PKLot-mini desde HuggingFace. Si ya está en `data/`, se omite la descarga.


In [4]:
dataset_path = CONFIG['dataset']['path']
already_downloaded = (os.path.isdir(os.path.join(dataset_path, 'images'))
                      and os.path.isfile(os.path.join(dataset_path, 'annotations.json')))

if already_downloaded:
    print(f"Meta-PKLot-mini ya descargado en {dataset_path}, se omite la descarga")
else:
    snapshot_download(
        repo_id=CONFIG['dataset']['repo'],
        repo_type='dataset',
        local_dir=dataset_path,
        token=os.environ.get('HF_TOKEN'),
    )

num_files = 0
total_bytes = 0

for root, _, files in os.walk(dataset_path):
    num_files += len(files)
    total_bytes += sum(os.path.getsize(os.path.join(root, f)) for f in files)

print(f"Dataset size: {total_bytes / 1024**3:.2f} GB ({num_files} files)")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5006 files:   0%|          | 0/5006 [00:00<?, ?it/s]

Dataset size: 1.24 GB (15021 files)


### Cargar el dataset con la libreria 

In [6]:
sys.path.insert(0, os.path.join(dataset_path, 'tools'))
from meta_pklot_mini import MetaPKLotMini

ds = MetaPKLotMini(dataset_path)

print(ds)
print("Datasets:", ds.datasets)
print("Climas:", ds.climates)
print("Condiciones:", ds.conditions)

MetaPKLotMini(root='/content/data/Meta-PKLot-mini', photos=4993)
Datasets: ('PKLot', 'CNRPark-EXT', 'ACPDS', 'ACMPS')
Climas: ('Overcast', 'Rainy', 'Sunny', 'Fog')
Condiciones: ('Glare', 'Occlusion_car', 'Occlusion_tree', 'Perspective_distortion')


# 3. Conjunto Test
Leemos el conjunto de Test

In [ ]:
# Leemos los IDs de las imágenes de test
url_test = f"{CONFIG['test']['url']}/{CONFIG['test']['file']}"
test_ids = pd.read_csv(url_test, header=None)[0].tolist()

# Buscar en el dataset la foto que corresponde a cada ID y creamos para cada imagen el objeto Pohto
test_photos = [ds.photo(i) for i in test_ids]

# Juntamos todas las plazas de esas fotos en una lista
test_spots = [s for p in test_photos for s in p.spots]

print("Fotos de test:", len(test_photos))
print("Plazas de test:", len(test_spots))

Fotos de test: 293
Plazas de test: 11236


# 4. Modelos preentrenados

## 4.1. Modelo 1: YOLO11m
YOLO es una red neuronal que se está entrenada con el dataset COCO que contiene alrededor de 120,000 imágenes con 80 tipos de clases. En este caso, sabe detectar coches ya que incluye en sus categorias la de vehicule_ids. Dentro de las 80 clases, nos quedamos con los 3 tipo sde vehículos que puede detectar: car, bus y truck.

In [14]:
# Cargar  modelo 
yolo = YOLO(CONFIG['models']['yolo11m']['weights'])

# IDs de COCO que corresponden a nuestras clases de vehículo
vehicle_ids = [i for i, name in yolo.names.items()
               if name in CONFIG['inference']['vehicle_classes']]
print("Clases de vehículo:", {i: yolo.names[i] for i in vehicle_ids})



Clases de vehículo: {2: 'car', 5: 'bus', 7: 'truck'}


Para cada imagen en test, esta la vamos a pasar por la red y dicha red nos da como salida: las cajas con coordenadas (x1, y1, x2 ,y2) que indica donde está el objeto detectado (la parte de regresión) y  puntuaciones asociadas a una categoría (parte de clasificación). Dado que hemos incluido el umbral de confianza, la red descarta aquellas detecciones con confianza menor al score_threshold (25%).

In [ ]:
detecciones = []   # un coche detectado por fila

for photo in test_photos: # Recorremos las imágenes de test

    # Pasar la imagen por YOLO
    result = yolo.predict(photo.image(),
                          imgsz=CONFIG['models']['yolo11m']['imgsz'],
                          conf=CONFIG['inference']['score_threshold'],
                          classes=vehicle_ids,
                          verbose=False)[0]

    # Guardar cada coche detectado en la imagen
    for caja, confianza, clase in zip(result.boxes.xyxy.cpu().numpy(),
                                      result.boxes.conf.cpu().numpy(),
                                      result.boxes.cls.cpu().numpy()):
        x1, y1, x2, y2 = caja

        detecciones.append({
            'image_id': photo.id,
            'clase': yolo.names[int(clase)],
            'confianza': float(confianza),
            'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
        })

detecciones_yolo = pd.DataFrame(detecciones) # Guarda en cada fila un coche 
print("Coches detectados en total:", len(detecciones_yolo))

Coches detectados en total: 8160


Dado que la red nos ha dado como salida cajas que indican la posición de donde está el objeto que la red a detectado, tenemos que hacer el puente de coches a plazas para poder comparar con las etiquetas que son en realidad plazas.
- Predicción: caja de coches
- Verdad: etiqeutas de plazas

In [ ]:
from shapely.geometry import Polygon, box

def calcular_overlap(plaza, coches): # Para cada plaza de una imagen
    """Devuelve qué fracción de la plaza tapa el coche que más la cubre
    y la confianza de ese coche."""
    poligono_plaza = Polygon(plaza.polygon)

    mejor_overlap = 0.0
    mejor_confianza = 0.0

    for _, coche in coches.iterrows():
        caja_coche = box(coche['x1'], coche['y1'], coche['x2'], coche['y2'])

        # Área común entre plaza y coche, dividida entre el área de la plaza
        overlap = poligono_plaza.intersection(caja_coche).area / poligono_plaza.area

        if overlap > mejor_overlap:
            mejor_overlap = overlap
            mejor_confianza = coche['confianza']

    return mejor_overlap, mejor_confianza

In [18]:
filas = []

for photo in test_photos:

    # Coches que YOLO detectó en una imagen
    coches = detecciones_yolo[detecciones_yolo['image_id'] == photo.id]

    # Para cada plaza de la imagen, calcular cuánto la tapa un coche
    for plaza in photo.spots:
        overlap, confianza = calcular_overlap(plaza, coches)
        filas.append({
            'image_id': photo.id,
            'spot_id': plaza.id,
            'real': plaza.label,            # etiqueta correcta: 'empty' / 'occupied'
            'overlap': overlap,
            'confianza': confianza,
        })

plazas_yolo = pd.DataFrame(filas)
print("Plazas:", len(plazas_yolo))
plazas_yolo.head()

Plazas: 11236


,image_id,spot_id,real,overlap,confianza
0,3401,184343,occupied,0.764027,0.967380
1,3401,184344,occupied,0.711381,0.925848
2,3401,184345,occupied,0.677155,0.921359
3,3401,184346,occupied,0.895217,0.888222
4,3401,184347,occupied,0.955809,0.944183


Una vez que tenemos para cada plaza del conjunto test la información de la fracción de la plaza que tapa el coche y la confianza de que haya un coche, entonces podemos decidir si una plaza está ocupada o libre para las predicciones.

In [19]:
umbral = CONFIG['occupancy']['min_overlap']  

plazas_yolo['prediccion'] = np.where(plazas_yolo['overlap'] >= umbral, 'occupied', 'empty')
plazas_yolo.head()

,image_id,spot_id,real,overlap,confianza,prediccion
0,3401,184343,occupied,0.764027,0.967380,occupied
1,3401,184344,occupied,0.711381,0.925848,occupied
2,3401,184345,occupied,0.677155,0.921359,occupied
3,3401,184346,occupied,0.895217,0.888222,occupied
4,3401,184347,occupied,0.955809,0.944183,occupied


# 5. Evaluación de Resultados

## 5.1.  Evaluación de resultados con YOLO

In [28]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, ConfusionMatrixDisplay

real = plazas_yolo['real']
pred = plazas_yolo['prediccion']
clases = ['empty', 'occupied']

# 1. Métricas en una tabla de una fila
precision, recall, f1, _ = precision_recall_fscore_support(real, pred, labels=clases)

metricas = pd.DataFrame([{
    'modelo': 'yolo11m',
    'test': CONFIG['test']['file'],
    'umbral_overlap': CONFIG['occupancy']['min_overlap'],
    'accuracy': accuracy_score(real, pred),
    'precision_empty': precision[0], 'recall_empty': recall[0], 'f1_empty': f1[0],
    'precision_occupied': precision[1], 'recall_occupied': recall[1], 'f1_occupied': f1[1],
}])


# Matriz de confusión en CSV
matriz_confusion_yolo = pd.crosstab(real, pred, rownames=['Real'], colnames=['Predicción'])



In [31]:
matriz_confusion_yolo

Predicción,empty,occupied
Real,,
empty,4114,1746
occupied,123,5253


YOLO detecta muy bien los coches. De las 5376 plazas ocupadas solo ha fallado en 123 casos.  La detección de vehículos funciona muy bien.
El problema son las plazas libres. De las plazas libres alrededor del 30%  se marcan como ocupadas.

### Guardar datos relevantes de modelo preentrenado YOLO

In [29]:
detecciones_yolo.to_csv(os.path.join(OUTPUT, 'detecciones_yolo11m.csv'), index=False)
plazas_yolo.to_csv(os.path.join(OUTPUT, 'plazas_yolo11m.csv'), index=False)

# Guardar las métricas de evaluación
metricas.to_csv(os.path.join(OUTPUT, 'metricas_yolo11m.csv'), index=False)
matriz_confusion_yolo.to_csv(os.path.join(OUTPUT, 'matriz_confusion_yolo11m.csv'))